# Chapter 3 (hadronic) — Notebook 2: Adding the b-jet

**Goals**

- Combine the best light-jet pair with the **hadronic-side** b-jet (the one farther from the lepton).

In [ ]:
%matplotlib inline
import awkward as ak
import numpy as np
import matplotlib.pyplot as plt

from topmass import io, kinematics, selection, plotting, fitting, neutrino, pairing, weights, style
from topmass.constants import M_W, M_TOP

In [ ]:
io.setup()                                  # select release 2025e-13tev-beta
samples = io.build_samples()                # skim '3J1LMET30', https
events = io.load_process('ttbar', samples, fraction=0.1)
print('Number of events:', len(events))

In [ ]:
cuts = selection.SemilepCuts(n_jets_min=4, n_bjets_min=2)
events = events[selection.semilep_preselection(events, cuts)]

lep = kinematics.leading_lepton(events)
jets = kinematics.jet_vectors(events)
is_b = events.jet_btag_quantile >= cuts.btag_quantile_min
b_jets = jets[is_b]
light = jets[~is_b]
keep = (ak.num(b_jets) >= 2) & (ak.num(light) >= 2)
lep, b_jets, light = lep[keep], b_jets[keep][:, :2], light[keep]

_, b_had = pairing.assign_bjets(b_jets, lep)
j1, j2, _ = pairing.best_W_pair(light)
m_top = (j1 + j2 + b_had).mass

plt.hist(ak.to_numpy(m_top), bins=60, range=(100, 350))
plt.axvline(172.5, color='red', label='generator $m_t$ = 172.5 GeV')
plt.xlabel(r'$m(jjb)$ [GeV]'); plt.legend()

## ✏️ Your turn 2.1

▶️ Change which b-jet you use and re-run.

The worked example used the **hadronic-side** b-jet (the one farther from the lepton). Switch
`USE_SIDE` to `'leptonic'` to deliberately use the *wrong* b-jet: the $m(jjb)$ peak should broaden
and shift away from 172.5 GeV — confirming that the b-jet assignment matters.

> **Stretch (optional):** run the cell once per `USE_SIDE` value without clearing the figure to
> overlay both.

In [ ]:
USE_SIDE = 'hadronic'    # ✏️ try 'leptonic' (the wrong b-jet)

b_lep, b_had = pairing.assign_bjets(b_jets, lep)
b_use = b_had if USE_SIDE == 'hadronic' else b_lep
j1, j2, _ = pairing.best_W_pair(light)
m_top = (j1 + j2 + b_use).mass

plt.hist(ak.to_numpy(m_top), bins=60, range=(100, 350), histtype='step', label=f'{USE_SIDE} b-jet')
plt.axvline(172.5, color='red', label='generator $m_t$ = 172.5 GeV')
plt.xlabel(r'$m(jjb)$ [GeV]'); plt.ylabel('Events'); plt.legend()